In [ ]:
# 실습에 필요한 패키지 설치 (최초 1회 실행)
!pip install -q openai python-dotenv httpx


# 1. 환경 설정 및 핵심 라이브러리

LLM API를 활용한 데이터 생성 및 증강 작업을 진행할 예정이다.
API 키를 안전하게 관리하고, 효율적인 통신을 위한 기본 설정을 먼저 구성한다.

## 1-1. 환경 변수

### 1-1-1. 환경 변수란?

- 운영체제가 프로그램 실행 환경에 제공하는 **전역 설정값**
- 프로그램은 이 변수를 통해 API 키, DB 비밀번호 등 민감한 정보를 **코드 외부에서** 읽어올 수 있다

### 1-1-2. 환경 변수 설정 이유

1. **보안**: API KEY를 코드에 직접 작성하면 GitHub 등에 공유 시 유출 위험이 있다. 환경 변수를 통해 민감 정보를 코드와 분리하여 관리한다.
2. **유연성**: 개발/테스트/배포 환경마다 다른 API KEY를 사용할 때, 코드를 수정하지 않고 환경 변수 값만 변경하면 된다.

### 1-1-3. .env 파일 생성 방법

이 노트북과 **같은 폴더**에 `.env` 파일을 생성하고, 아래 내용을 작성한다.

```
UPSTAGE_API_KEY=여기에_실제_API키를_붙여넣기
```

> **💡 .env 파일이란?**
>
> 키=값 형태로 환경 변수를 저장하는 텍스트 파일이다.
> `python-dotenv` 라이브러리가 이 파일을 읽어 운영체제의 환경 변수로 등록해 준다.
> `.gitignore`에 `.env`를 추가하면 GitHub에 실수로 올리는 것을 방지할 수 있다.


In [ ]:
from dotenv import load_dotenv
from os import getenv

# .env 파일에서 환경 변수를 로드
# 왜 load_dotenv()를 호출하는가?
# → .env 파일에 적힌 키=값 쌍을 운영체제의 환경 변수로 등록한다.
#   이후 getenv()로 해당 값을 파이썬 코드 안에서 꺼내 쓸 수 있다.
load_dotenv()  # 현재 디렉토리의 .env 파일을 자동으로 찾아 로드

# 등록된 환경 변수에서 API 키를 가져온다
UPSTAGE_API_KEY = getenv('UPSTAGE_API_KEY')

if UPSTAGE_API_KEY:
    print('API 키 로드 성공!')
else:
    print('ERROR: .env 파일에 UPSTAGE_API_KEY가 설정되지 않았습니다.')
    print('이 노트북과 같은 폴더에 .env 파일을 생성하고 API 키를 입력하세요.')


## 1-2. Upstage API 기본 사용 방법

### 1-2-1. openai 라이브러리 사용

Upstage API는 OpenAI API와 요청/응답 구조가 **호환**된다.
따라서 `openai` 라이브러리를 그대로 사용하되, 요청을 보내는 서버 주소(`base_url`)만 Upstage API 엔드포인트로 변경하면 된다.

> **💡 왜 openai 라이브러리를 쓰는가?**
>
> OpenAI가 만든 API 통신 규격(Chat Completions API)이 업계 표준이 되었다.
> Upstage, Anthropic 등 다른 회사들도 이 규격에 맞춰 API를 제공하므로,
> `openai` 라이브러리 하나로 여러 회사의 LLM을 사용할 수 있다.
> 바꿔야 할 것은 `base_url`과 `api_key`뿐이다.


In [ ]:
from openai import OpenAI

# ========== Upstage API 클라이언트 생성 ==========
# OpenAI 클래스에 Upstage의 주소와 키를 넣으면 Upstage LLM과 통신할 수 있다.
client = OpenAI(
    api_key=UPSTAGE_API_KEY,                   # .env에서 읽어온 API 키
    base_url='https://api.upstage.ai/v1',      # Upstage 서버 주소
)

# ========== 채팅 요청 보내기 ==========
# client.chat.completions.create: 채팅 기반 LLM에 요청을 보내는 메서드
response = client.chat.completions.create(
    model='solar-pro3',          # 사용할 LLM 모델 이름
    messages=[                    # 대화 내용 (역할 + 내용 쌍의 리스트)
        {
            'role': 'user',       # 'user': 사용자가 보내는 메시지
            'content': '안녕? 넌 이름이 뭐니?',
        }
    ],
    # stream=True: 응답을 한 번에 받지 않고, 생성되는 대로 조각(chunk)씩 실시간 수신
    # → 사용자가 타이핑하듯 글자가 하나씩 나타나는 효과
    stream=True,
)

# ========== 스트리밍 응답 출력 ==========
# 각 조각(chunk)에서 새로운 텍스트가 있으면 이어서 출력한다
for chunk in response:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end='')  # end='' → 줄바꿈 없이 이어서 출력

# 참고: stream=False로 설정하면 전체 응답을 한 번에 받을 수 있다
# print(response.choices[0].message.content)


## 1-3. HTTPX (비동기 통신)

### 1-3-1. HTTPX란?

- Python용 HTTP 클라이언트 라이브러리
- 기존 `requests`와 사용법이 유사하면서 **비동기(Asynchronous)** 통신을 지원하는 것이 특징

> **💡 비동기 통신이란?**
>
> 여러 개의 API 요청을 보낼 때, 하나의 요청이 끝날 때까지 **기다리지 않고**
> 여러 요청을 **동시에 병렬로** 처리하는 방식이다.
>
> 예: 데이터 100건을 생성해야 할 때
> - 동기: 1건 요청 → 응답 대기 → 다음 1건 요청 → ... (순차 처리, 느림)
> - 비동기: 100건 동시 요청 → 결과를 모아서 처리 (병렬 처리, 빠름)

### 1-3-2. 핵심 키워드

- `async def`: 이 함수가 **비동기 함수**임을 선언. 내부에서 `await` 사용 가능
- `await`: 비동기 작업이 완료될 때까지 기다리되, **다른 비동기 작업은 계속 실행**되도록 함
  - 즉, "기다리는 동안 다른 일 먼저 하고 있어!"라는 의미
- `asyncio.gather(*tasks)`: 여러 비동기 작업을 **동시에 실행**하고 모든 결과를 모아서 반환


In [ ]:
import httpx
import asyncio

# ========== 비동기 API 호출 함수 ==========
# 왜 별도의 함수로 분리하는가?
# → 여러 프롬프트를 동시에 요청하기 위해, 각 요청을 독립적인 "작업(Task)"으로 만들어야 한다.
#   이 함수 하나가 하나의 작업에 해당한다.
async def call_chat_completion(url, headers, payload):
    print('    call_chat_completion 시작')
    # httpx.AsyncClient: 비동기 HTTP 요청을 보내는 클라이언트
    # timeout=30.0: 30초 안에 응답이 없으면 에러 발생
    async with httpx.AsyncClient(timeout=30.0) as client:
        # await: 응답이 올 때까지 비동기적으로 대기
        # → 이 요청이 기다리는 동안 다른 요청이 동시에 실행될 수 있다
        response = await client.post(url, headers=headers, json=payload)
        response.raise_for_status()  # HTTP 오류(4xx, 5xx) 발생 시 예외
        data = response.json()
        return data['choices'][0]['message']['content']


In [ ]:
# ========== 여러 작업을 동시에 실행하는 메인 함수 ==========
async def request(tasks):
    print('모든 요청 동시 실행 시작')
    # asyncio.gather: 리스트의 모든 비동기 작업을 동시에 실행하고
    # 모든 작업이 완료될 때까지 기다린 후 결과를 리스트로 반환
    results = await asyncio.gather(*tasks)
    print('모든 요청 완료\n')

    for i, res in enumerate(results, 1):
        print(f'Response {i}')
        print(res)
        print('=' * 40)


# ========== API 설정 ==========
url = 'https://api.upstage.ai/v1/chat/completions'
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {UPSTAGE_API_KEY}',
}

# ========== 동시에 보낼 프롬프트 준비 ==========
prompts = [
    '농담 하나만 해 줘',
    '프랑스의 수도는 어디야?',
]

# 각 프롬프트를 비동기 작업(Task)으로 변환
tasks = []
for prompt in prompts:
    payload = {
        'model': 'solar-pro3',
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,  # 비동기에서는 스트리밍 대신 한 번에 받는다
    }
    # 함수를 호출하지만 await가 없으므로 아직 실행되지 않는다
    # → "실행 대기 상태"의 코루틴 객체만 만들어진다
    print(f'태스크 생성: "{prompt}"')
    tasks.append(call_chat_completion(url, headers, payload))
    print('  → 아직 API 호출 전 (대기 상태)')

# ========== 모든 태스크 동시 실행 ==========
# Jupyter Notebook에서는 await를 직접 사용할 수 있다
await request(tasks)


## 1-4. JSON (JavaScript Object Notation)

- 데이터를 저장하거나 주고받을 때 사용하는 **가볍고 사람이 읽기 쉬운** 데이터 형식
- 파이썬의 딕셔너리와 유사한 `"키": 값` 형태로 구성된다
- 대부분의 프로그래밍 언어와 API 통신에서 **표준**처럼 사용된다

> **💡 왜 JSON을 강제하는가?**
>
> LLM은 기본적으로 "자유로운 텍스트"로 응답한다.
> 하지만 합성 데이터를 만들 때는 **일관된 구조**가 필요하다.
> `response_format` 옵션으로 JSON 스키마를 지정하면,
> LLM이 반드시 해당 구조에 맞춰 응답하도록 강제할 수 있다.


In [ ]:
import json

# ========== JSON 응답 형식 정의 ==========
# LLM에게 "이 구조에 맞춰서 대답해"라고 강제하는 설정이다.
response_format = {
    'type': 'json_schema',        # 응답 형식을 JSON 스키마로 지정
    'json_schema': {
        'name': '수도 정보',       # 스키마 이름 (디버깅용)
        'strict': True,            # 엄격 모드: 스키마에 안 맞으면 에러
        'schema': {
            'type': 'object',      # 최상위 타입: 딕셔너리(객체)
            'properties': {        # 포함될 속성(키) 정의
                'capital': {'type': 'string'},
                'translation': {'type': 'string', 'description': '수도의 영어 번역'},
            },
            'required': ['capital', 'translation'],  # 반드시 포함해야 할 키
        },
    },
}

# ========== JSON 형식으로 응답 받기 ==========
response = client.chat.completions.create(
    model='solar-pro3',
    messages=[{'role': 'user', 'content': '한국의 수도는 어디야?'}],
    response_format=response_format,  # JSON 스키마 적용
)

# LLM 응답은 JSON 형식의 "문자열"이므로, 파이썬 딕셔너리로 변환해야 한다
# json.loads: JSON 문자열 → 파이썬 딕셔너리
structured_dictionary = json.loads(response.choices[0].message.content)

print('구조화된 응답:')
for key, value in structured_dictionary.items():
    print(f'  {key}: {value}')


- 더 구체적인 response_format 사용 방법은 아래 링크 참고
- [OpenAI Structured Model Outputs](https://platform.openai.com/docs/guides/structured-outputs?lang=python)
